# Biomarqueurs vocaux de la dépression : Androids et DAIC-WOZ

Ce notebook applique une même chaîne de traitement à deux corpus de référence, le corpus Androids et le DAIC-WOZ, et compare ce qu'elle donne sur chacun.

La chaîne consiste à découper les enregistrements en tours de parole, en extraire les 88 descripteurs eGeMAPS avec openSMILE, on centre et réduit, puis on classe avec un SVM à noyau gaussien. L'évaluation est une validation croisée répétée et stratifiée, toujours groupée par locuteur.

Tous les résultats utilisent `random_state = 0`.

## Données

Le corpus Androids est disponible auprès de ses auteurs. Le DAIC-WOZ demande la signature d'un accord d'utilisation auprès de l'USC ICT, il n'est donc pas inclus ici. Les chemins attendus sont indiqués dans la cellule de configuration.

Les descripteurs sont mis en cache dans des fichiers `.npz` pour éviter de réextraire l'audio à chaque exécution. Si les caches sont absents, les cellules d'extraction les reconstruisent, ce qui prend un certain temps.

In [ ]:
import os, glob, warnings
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import plotly.express as px
import opensmile
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, StratifiedGroupKFold,
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix,
)
warnings.filterwarnings("ignore")

C_HC, C_PT, C_A, C_B = "#0072B2", "#D55E00", "#009E73", "#CC79A7"
RNG = 0

In [ ]:
ANDROIDS_BASE = "androids_corpus/Androids-Corpus"
ANDROIDS_IT_CLIPS = os.path.join(ANDROIDS_BASE, "Interview-Task", "audio_clip")
ANDROIDS_RT_AUDIO = os.path.join(ANDROIDS_BASE, "Reading-Task", "audio")
DAIC_CHUNKS = "patients_chunks"
DAIC_LABELS = "labels.csv"
AND_CACHE  = "and_features.npz"
DAIC_CACHE = "daic_features.npz"
CACHE_ALL  = "features_cache.npz"
FIGDIR     = "rapport_figures"

os.makedirs(FIGDIR, exist_ok=True)

## Vérification des corpus

Les deux corpus doivent être placés dans le répertoire courant. Ils ne sont pas fournis avec ce
notebook.

In [ ]:
assert os.path.isdir(ANDROIDS_BASE), \
    "Androids Corpus introuvable, placer le dossier androids_corpus/ dans le répertoire courant"
print("Androids Corpus present.")

assert os.path.isdir(DAIC_CHUNKS), \
    "DAIC-WOZ introuvable, placer le dossier patients_chunks/ dans le répertoire courant"
print(f"Dossiers patients : {len(os.listdir(DAIC_CHUNKS))}")
print(f"Fichiers WAV total : {len(glob.glob(DAIC_CHUNKS + '/*/*.wav'))}")

labels_df = pd.read_csv(DAIC_LABELS)
print(f"Participants : {len(labels_df)}")
print(f"Depressifs   : {labels_df['PHQ8_Binary'].sum()}")
print(f"Controles    : {(labels_df['PHQ8_Binary'] == 0).sum()}")

## Fonctions communes

Extraction des descripteurs et calcul des métriques.

In [ ]:
smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

def extract_functionals(audio_path):
    try:
        return smile.process_file(audio_path).values.flatten()
    except Exception:
        return None

def clean_nan(X, y, *extra):
    n_before = len(X)
    mask = ~np.isnan(X).any(axis=1)
    n_dropped = n_before - int(mask.sum())
    if n_dropped:
        print(f"  clean_nan : {n_dropped} supprimé(s) sur {n_before} ({n_dropped/n_before:.1%})")
    out = [X[mask], y[mask]] + [e[mask] for e in extra]
    return tuple(out) if extra else (X[mask], y[mask])

def evaluate(y_true, y_pred, y_score=None):
    """Accuracy, F1 macro, UAR et AUC. La F1 de la classe positive est calculée en plus."""
    d = {
        "F1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "UAR": balanced_accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1_pos": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
    }
    if y_score is not None:
        try:
            d["AUC"] = roc_auc_score(y_true, y_score)
        except ValueError:
            d["AUC"] = float("nan")
    return d

# colonnes affichées dans les tableaux
REPORT_COLS = ["Accuracy", "F1_macro", "UAR", "AUC"]

def show(df_cv, title=""):
    if title: print(title)
    for c in REPORT_COLS:
        print(f"  {c}: {df_cv[c].mean():.3f} +/- {df_cv[c].std():.3f}")
    print()

# Le corpus Androids

On commence par Androids, sur lequel la chaîne fonctionne. Cela permet de vérifier le pipeline
avant de le confronter au DAIC-WOZ. Le corpus réunit 118 locuteurs italiens enregistrés en centre
de soin, avec deux tâches, une lecture à voix haute et un entretien. Les étiquettes viennent d'un
diagnostic de psychiatre selon le DSM-5.

In [ ]:
def load_androids():
    """Charge les clips Interview d'Androids (features + pid + durée). Utilise le cache si présent."""
    if os.path.exists(AND_CACHE):
        d = np.load(AND_CACHE, allow_pickle=True)
        print(f"Androids (cache) : {d['X'].shape[0]} clips")
        return d["X"], d["y"], d["pids"], d["durs"], d["fnames"]
    clip_dirs = sorted(glob.glob(os.path.join(ANDROIDS_IT_CLIPS, "*")))
    tasks = []
    for cdir in clip_dirs:
        if not os.path.isdir(cdir):
            continue
        base = os.path.basename(cdir)
        label = 0 if base.split("_")[1][0] == "C" else 1
        for wf in sorted(glob.glob(os.path.join(cdir, "*.wav"))):
            tasks.append((wf, label, base))
    res = Parallel(n_jobs=-1)(delayed(lambda w: (extract_functionals(w), sf.info(w).duration))(w)
                              for w, _, _ in tasks)
    X, y, pids, durs, fnames = [], [], [], [], []
    for (f, dur), (wf, lab, pid) in zip(res, tasks):
        if f is not None:
            X.append(f); y.append(lab); pids.append(pid); durs.append(dur); fnames.append(wf)
    X, y = np.array(X), np.array(y)
    return X, y, np.array(pids), np.array(durs), np.array(fnames)

X_and_chunks, y_and_chunks, and_pids, and_durs, and_fnames = load_androids()
X_and_chunks, y_and_chunks, and_pids, and_durs = clean_nan(
    X_and_chunks, y_and_chunks, and_pids, and_durs)
print(f"{len(np.unique(and_pids))} locuteurs, {X_and_chunks.shape[1]} descripteurs")

## Agrégation early fusion

Les descripteurs sont calculés par tour de parole, mais on cherche à classer le locuteur. En early
fusion, on moyenne les descripteurs de tous ses tours pour obtenir un vecteur unique par locuteur.

In [ ]:
def early_fusion_features(X_chunks, y_chunks, pids):
    """Un vecteur moyen par locuteur."""
    upids = np.unique(pids)
    X = np.array([X_chunks[pids == p].mean(axis=0) for p in upids])
    y = np.array([y_chunks[pids == p][0] for p in upids])
    return X, y, upids

def within_corpus_cv(X, y, title="", n_splits=5, n_repeats=10):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=RNG)
    rows = []
    for tr, te in rskf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = SVC(class_weight="balanced", probability=True, random_state=RNG)
        clf.fit(sc.transform(X[tr]), y[tr])
        p = clf.predict(sc.transform(X[te]))
        s = clf.predict_proba(sc.transform(X[te]))[:, 1]
        rows.append(evaluate(y[te], p, s))
    return pd.DataFrame(rows)

X_and, y_and, _ = early_fusion_features(X_and_chunks, y_and_chunks, and_pids)
print(f"Androids early : {X_and.shape[0]} locuteurs, {int(y_and.sum())} PT / {int((y_and==0).sum())} HC")
df_and = within_corpus_cv(X_and, y_and)
show(df_and, "Androids Interview (early fusion, F1 macro)")

### Résultats intra-corpus

Cette cellule ne traite que la tâche d'entretien.

In [ ]:
def intra_row(name, df_cv):
    return {"Corpus": name, **{c: f"{df_cv[c].mean():.3f} ± {df_cv[c].std():.3f}" for c in REPORT_COLS}}

androids_table = pd.DataFrame([
    intra_row("Androids Interview (entretien)", df_and),
])
androids_table

## La tâche de lecture

Androids comporte une seconde tâche, la lecture à voix haute d'une fable d'Ésope. Un fichier par
locuteur, contre plusieurs clips en entretien.

Les identifiants ne suivent pas le même format d'une tâche à l'autre, `01_CF56_1` en entretien et
`01_C` en lecture. On les ramène donc à leur préfixe `nn_X`, qui désigne le locuteur et sa
condition.

In [ ]:
RT_CACHE = "reading_features.npz"

def speaker_key(name):
    """01_CF56_1 -> 01_C ; 01_C -> 01_C"""
    parts = os.path.basename(str(name)).split("_")
    return f"{parts[0]}_{parts[1][0]}"

def load_reading():
    """Charge la tâche de lecture (un fichier par locuteur). Utilise le cache si présent."""
    if os.path.exists(RT_CACHE):
        d = np.load(RT_CACHE, allow_pickle=True)
        print(f"Reading-Task (cache) : {d['X'].shape[0]} fichiers")
        return d["X"], d["y"], d["sid"]
    files = [(f, 0) for f in sorted(glob.glob(os.path.join(ANDROIDS_RT_AUDIO, "HC", "*.wav")))] + \
            [(f, 1) for f in sorted(glob.glob(os.path.join(ANDROIDS_RT_AUDIO, "PT", "*.wav")))]
    res = Parallel(n_jobs=-1)(delayed(extract_functionals)(f) for f, _ in files)
    X, y, sid = [], [], []
    for (f, lab), fe in zip(files, res):
        if fe is not None:
            X.append(fe); y.append(lab); sid.append(speaker_key(f))
    X, y, sid = np.array(X), np.array(y), np.array(sid)
    np.savez_compressed(RT_CACHE, X=X, y=y, sid=sid)
    return X, y, sid

X_rt, y_rt, rt_sid = load_reading()
X_rt, y_rt, rt_sid = clean_nan(X_rt, y_rt, rt_sid)
print(f"{len(np.unique(rt_sid))} locuteurs, {int(y_rt.sum())} PT / {int((y_rt == 0).sum())} HC")

df_rt = within_corpus_cv(X_rt, y_rt, "Androids Reading (intra)")

## La condition combinée

On moyenne par locuteur les descripteurs des deux tâches. Seuls les locuteurs présents dans les
deux sont retenus, puisque moyenner deux tâches n'a pas de sens quand une seule est disponible.

In [ ]:
# vecteur moyen par locuteur, dans chaque tâche
itv = {}
for p in np.unique(and_pids):
    m = and_pids == p
    itv[speaker_key(p)] = (X_and_chunks[m].mean(axis=0), y_and_chunks[m][0])

rea = {}
for s in np.unique(rt_sid):
    m = rt_sid == s
    rea[s] = (X_rt[m].mean(axis=0), y_rt[m][0])

common = sorted(set(itv) & set(rea))
desaccord = [s for s in common if itv[s][1] != rea[s][1]]
print(f"locuteurs : entretien={len(itv)}, lecture={len(rea)}, communs={len(common)}")
if desaccord:
    print("étiquettes discordantes :", desaccord)

X_comb = np.array([(itv[s][0] + rea[s][0]) / 2 for s in common])
y_comb = np.array([itv[s][1] for s in common])
print(f"classes : {int((y_comb == 0).sum())} HC / {int(y_comb.sum())} PT")

df_comb = within_corpus_cv(X_comb, y_comb, "Androids Combiné (intra)")

### Les trois conditions

In [ ]:
androids_df = pd.DataFrame([
    intra_row("Lecture",   df_rt),
    intra_row("Entretien", df_and),
    intra_row("Combiné",   df_comb),
]).rename(columns={"Corpus": "Tâche"})

androids_df

## Métadonnées d'Androids

Sexe, âge et niveau d'éducation, encodés dans les noms de fichiers.

In [ ]:
# métadonnées encodées dans les noms de fichiers : nn_XGmm_t
import re
spk = {}
for f in glob.glob(os.path.join(ANDROIDS_IT_CLIPS, "*")):
    m = re.match(r"(\d+)_([PC])([MFX])(\d+)_(\d)", os.path.basename(f))
    if m:
        nn, cond, g, age, edu = m.groups()
        spk[f"{nn}_{cond}"] = (cond, g, int(age), int(edu))
meta = pd.DataFrame([(k, *v) for k, v in spk.items()],
                    columns=["sid", "cond", "gender", "age", "edu"])
meta["groupe"] = meta.cond.map({"P": "Dépressifs (PT)", "C": "Témoins (HC)"})
print(meta.groupby("groupe").agg(n=("sid", "size"),
      femmes=("gender", lambda s: (s == "F").sum()),
      hommes=("gender", lambda s: (s == "M").sum()),
      age_moy=("age", "mean"), edu_moy=("edu", "mean")).round(2))
print("\nLes groupes sont appariés en âge et en niveau d'éducation.")

# Le corpus DAIC-WOZ

Le DAIC-WOZ réunit 189 entretiens en anglais, conduits par une interface animée pilotée à distance.
Les étiquettes proviennent d'un auto-questionnaire, le PHQ-8, avec un seuil à 10.

Les segments chargés ici ont déjà été découpés et nettoyés par le script de segmentation, qui corrige les décalages entre audio et  ranscription, retire les bips de synchronisation, les interruptions et les passages anonymisés.

Les étiquettes sont relues depuis `labels.csv`.
Attention : le participant 409 était marqué non dépressif dans le corpus orginal alors que son score PHQ-8 vaut 10.

In [ ]:
def load_daic_chunks():
    if os.path.exists(DAIC_CACHE):
        d = np.load(DAIC_CACHE, allow_pickle=True)
        print(f"DAIC-WOZ (cache) : {d['X'].shape[0]} chunks")
        return d["X"], d["y"], d["pids"], d["durs"], d["fnames"]
    labels = pd.read_csv(DAIC_LABELS)
    tasks = []
    for pid, lab in zip(labels.Participant_ID, labels.PHQ8_Binary):
        for wf in sorted(glob.glob(os.path.join(DAIC_CHUNKS, str(pid), "*.wav"))):
            tasks.append((wf, lab, pid))
    res = Parallel(n_jobs=-1)(delayed(lambda w: (extract_functionals(w), sf.info(w).duration))(w)
                              for w, _, _ in tasks)
    X, y, pids, durs, fnames = [], [], [], [], []
    for (f, dur), (wf, lab, pid) in zip(res, tasks):
        if f is not None:
            X.append(f); y.append(lab); pids.append(pid); durs.append(dur); fnames.append(wf)
    return np.array(X), np.array(y), np.array(pids), np.array(durs), np.array(fnames)

Xc, yc, pid, dur, fn = load_daic_chunks()
Xc, yc, pid, dur = clean_nan(Xc, yc, pid, dur)

labels = pd.read_csv(DAIC_LABELS).set_index("Participant_ID")["PHQ8_Binary"].to_dict()
yc = np.array([labels[int(p)] for p in pid])
print(f"{Xc.shape[0]} chunks, {len(np.unique(pid))} locuteurs, "
      f"{len(np.unique(pid[yc==1]))} dépressifs / {len(np.unique(pid[yc==0]))} témoins")

## Métadonnées DAIC-WOZ

Le corpus ne fournit que le sexe. La cellule suivante trace un exemple de découpage sur une
session complète.

In [ ]:
lab = pd.read_csv(DAIC_LABELS)
print("DAIC-WOZ : genre x étiquette")
print(pd.crosstab(lab.PHQ8_Binary, lab.Gender))
print("PHQ-8 moyen — dépressifs:", round(lab[lab.PHQ8_Binary==1].PHQ8_Score.mean(),1),
      "| témoins:", round(lab[lab.PHQ8_Binary==0].PHQ8_Score.mean(),1),
      "| seuil dépression >= 10")

In [ ]:
# Figure de chunking : forme d'onde d'une session, tours participant (chunks) vs Ellie (exclu)
# Reconstruite à partir des chunks .wav placés à leur position temporelle (Ellie a été retirée).
from matplotlib.patches import Patch
PID, SR, T0, T1 = 308, 16000, 55.0, 103.0
tr = pd.read_csv(f"transcript/{PID}_TRANSCRIPT.csv", sep="\t")
tr.columns = [c.strip() for c in tr.columns]; tr["speaker"] = tr.speaker.astype(str).str.strip().str.lower()
chunks = sorted(glob.glob(f"patients_chunks/{PID}/*.wav"), key=lambda p: int(re.findall(r"_(\d+)\.wav", p)[0]))
part = tr[tr.speaker == "participant"].reset_index(drop=True)
n = int((T1 - T0) * SR); wave = np.zeros(n); kept = []
for i, (_, r) in enumerate(part.iterrows()):
    if r.stop_time < T0 or r.start_time > T1: continue
    a, _ = sf.read(chunks[i]); a = a.mean(1) if a.ndim > 1 else a
    i0 = int((r.start_time - T0) * SR); seg = a[:max(0, min(len(a), n - i0))]
    if 0 <= i0 < n: wave[i0:i0+len(seg)] = seg
    kept.append((r.start_time, r.stop_time, i+1))
ellie = [(r.start_time, r.stop_time) for _, r in tr[tr.speaker=="ellie"].iterrows() if r.stop_time>T0 and r.start_time<T1]
t = np.linspace(T0, T1, n); mx = np.abs(wave).max() or 1
fig, (ax, axl) = plt.subplots(2, 1, figsize=(11, 4.4), height_ratios=[3,1], sharex=True)
ax.plot(t, wave, lw=0.4, color=C_HC)
for s, e in ellie: ax.axvspan(max(s,T0), min(e,T1), color="#999", alpha=0.18)
for s, e, idx in kept:
    ax.axvline(s, color=C_PT, ls="--", lw=0.9); ax.axvline(e, color=C_PT, ls="--", lw=0.9)
ax.set_ylim(-mx*1.15, mx*1.25); ax.set_yticks([]); ax.set_ylabel("Amplitude")
ax.set_title(f"Chunking d'une session DAIC-WOZ (participant {PID})")
for s, e in ellie: axl.axvspan(max(s,T0), min(e,T1), ymin=.55, ymax=.95, color="#999", alpha=.55)
for s, e, idx in kept: axl.axvspan(s, e, ymin=.08, ymax=.48, color=C_HC, alpha=.75)
axl.text(T0+.2,.75,"Ellie (exclu)",fontsize=8,va="center"); axl.text(T0+.2,.28,"Participant (chunks)",fontsize=8,va="center")
axl.set_ylim(0,1); axl.set_yticks([]); axl.set_xlabel("Temps (s)"); axl.set_xlim(T0,T1)
ax.legend(handles=[Patch(color=C_HC,alpha=.75,label="Tour participant → 1 chunk"),
                   Patch(color="#999",alpha=.55,label="Tour d'Ellie → exclu"),
                   plt.Line2D([0],[0],color=C_PT,ls="--",label="Borne de découpe")], loc="upper right", fontsize=7)
fig.tight_layout(); fig.savefig("rapport_figures/fig_chunking_waveform.png", dpi=200); plt.show()

## Résultats bruts

Appliquée telle quelle en early fusion, la chaîne reste au niveau du hasard sur le DAIC-WOZ. C'est
le point de départ de la section suivante.

In [ ]:
X_daic, y_daic, _ = early_fusion_features(Xc, yc, pid)
print(f"DAIC early : {X_daic.shape[0]} patients, {int(y_daic.sum())} dépressifs")
df_daic = within_corpus_cv(X_daic, y_daic)
show(df_daic, "DAIC-WOZ brut (early fusion, F1 macro)")

# Tenter de combler l'écart

L'explication la plus simple de cet écart accuse le corpus, le DAIC-WOZ étant nettement plus dégradé
qu'Androids. Cette section teste cette hypothèse en retirant une à une les sources de dégradation,
pour voir si la performance remonte.

## Projection en composantes principales

Avant de classer, on regarde à quoi ressemblent les descripteurs. La projection ne sépare pas les
dépressifs des témoins. En revanche elle fait apparaître nettement des groupes de segments sans
parole, une ligne de silences et une zone de souffles peu bruyants.

Une version interactive est exportée en HTML.

In [ ]:
Xs = StandardScaler().fit_transform(Xc)
pca = PCA(n_components=2, random_state=RNG).fit(Xs)
Z = pca.transform(Xs)
ev = pca.explained_variance_ratio_ * 100
print(f"Variance expliquée : PC1 {ev[0]:.1f} %, PC2 {ev[1]:.1f} % (cumul {ev.sum():.1f} %)")

label = np.where(yc == 1, "PT (déprimé)", "HC (témoin)")
fig, ax = plt.subplots(figsize=(7.2, 5.4))
for name, col in [("HC (témoin)", C_HC), ("PT (déprimé)", C_PT)]:
    m = label == name
    ax.scatter(Z[m, 0], Z[m, 1], s=5, c=col, alpha=0.35, label=name, edgecolors="none")
ax.set_xlabel(f"PC1 ({ev[0]:.1f} % de variance)")
ax.set_ylabel(f"PC2 ({ev[1]:.1f} % de variance)")
ax.set_title("ACP des chunks DAIC-WOZ (eGeMAPS)")
ax.legend(markerscale=3); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig("rapport_figures/fig_pca_daic.png", dpi=200); plt.show()

# version interactive pour le site (wwwetu.utc.fr) -> images/PCA_DAIC-WOZ.html
os.makedirs("rapport_figures/interactive", exist_ok=True)
dfv = pd.DataFrame({"PC1": Z[:,0], "PC2": Z[:,1], "label": label, "durée (s)": dur.round(2), "locuteur": pid})
figp = px.scatter(dfv, x="PC1", y="PC2", color="label",
                  color_discrete_map={"HC (témoin)": C_HC, "PT (déprimé)": C_PT},
                  hover_data=["locuteur", "durée (s)"],
                  labels={"PC1": f"PC1 ({ev[0]:.1f} % variance)", "PC2": f"PC2 ({ev[1]:.1f} % variance)"})
figp.update_traces(marker=dict(size=4, opacity=0.5))
figp.write_html("rapport_figures/interactive/PCA_DAIC-WOZ.html", include_plotlyjs="cdn")

## Late fusion et découpage groupé par locuteur

En late fusion, chaque tour est classé séparément, puis on remonte au locuteur par vote majoritaire.

Le découpage se fait sur la liste des locuteurs, ce qui équivaut à un `StratifiedGroupKFold`. Tous
les tours d'un locuteur de test restent donc du côté test.

In [ ]:
def late_fusion_cv(X_chunks, y_chunks, patient_ids, durations,
                   min_duration=0.0, n_splits=5, n_repeats=10,
                   random_state=RNG, cap=4000):
    """cap : sous-échantillonnage des segments d'entraînement (vitesse) ; mettre None pour tout garder."""
    dur_mask = durations >= min_duration
    Xf, yf, pf = X_chunks[dur_mask], y_chunks[dur_mask], patient_ids[dur_mask]
    upids = np.unique(pf)
    pid2lab = {p: yf[pf == p][0] for p in upids}
    plabs = np.array([pid2lab[p] for p in upids])
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    rows = []
    for tr, te in rskf.split(upids, plabs):           # <-- split AU NIVEAU LOCUTEUR
        train = set(upids[tr]); test = upids[te]
        trm = np.isin(pf, list(train)); tem = ~trm
        Xtr_, ytr_ = Xf[trm], yf[trm]
        if cap and len(Xtr_) > cap:
            _i = np.random.RandomState(random_state).choice(len(Xtr_), cap, replace=False)
            Xtr_, ytr_ = Xtr_[_i], ytr_[_i]
        sc = StandardScaler().fit(Xtr_)
        clf = SVC(class_weight="balanced", random_state=random_state)
        clf.fit(sc.transform(Xtr_), ytr_)
        pred = clf.predict(sc.transform(Xf[tem])); sco = clf.decision_function(sc.transform(Xf[tem]))
        pte = pf[tem]; yt, yp, ys = [], [], []
        for p in test:
            pm = pte == p
            if pm.sum() == 0: continue
            yt.append(pid2lab[p]); yp.append(1 if pred[pm].mean() >= .5 else 0); ys.append(sco[pm].mean())
        rows.append(evaluate(np.array(yt), np.array(yp), np.array(ys)))
    return pd.DataFrame(rows)

df_daic_late = late_fusion_cv(Xc, yc, pid, dur, min_duration=0.0, n_repeats=10)
show(df_daic_late, "DAIC-WOZ late fusion (tous les segments, groupé par locuteur)")

### Effet de la fuite de locuteur

Si l'on découpe naïvement au niveau des tours avec un `StratifiedKFold`, des segments d'un même
locuteur se retrouvent en apprentissage et en test. Le modèle apprend alors à reconnaître le
locuteur plutôt que sa condition, et les scores montent nettement.

La cellule ci-dessous compare les deux découpages sur les mêmes données. L'écart mesure ce que
coûterait l'erreur.

In [ ]:
def chunk_cv(X, y, groups, splitter, n_repeats=1):
    rows = []
    for rep in range(n_repeats):
        for tr, te in splitter(rep).split(X, y, groups):
            sc = StandardScaler().fit(X[tr])
            clf = SVC(class_weight="balanced", random_state=RNG).fit(sc.transform(X[tr]), y[tr])
            pred = clf.predict(sc.transform(X[te])); sco = clf.decision_function(sc.transform(X[te]))
            rows.append(evaluate(y[te], pred, sco))
    return pd.DataFrame(rows)

# aucun filtre de durée : le balayage a montré qu'il ne change rien
leak = chunk_cv(Xc, yc, pid, lambda r: StratifiedKFold(5, shuffle=True, random_state=RNG+r), n_repeats=3)
grp  = chunk_cv(Xc, yc, pid, lambda r: StratifiedGroupKFold(5), n_repeats=1)
print("Fuite (StratifiedKFold sur chunks) :", {c: round(leak[c].mean(),3) for c in REPORT_COLS})
print("Correct (StratifiedGroupKFold)     :", {c: round(grp[c].mean(),3)  for c in REPORT_COLS})

fig, ax = plt.subplots(figsize=(6.4, 4.4))
x = np.arange(len(REPORT_COLS)); w = .35
ax.bar(x-w/2, [leak[c].mean() for c in REPORT_COLS], w, color=C_PT, label="StratifiedKFold sur chunks (fuite)")
ax.bar(x+w/2, [grp[c].mean()  for c in REPORT_COLS], w, color=C_HC, label="StratifiedGroupKFold (par locuteur)")
ax.axhline(.5, ls=":", c="grey"); ax.set_xticks(x); ax.set_xticklabels(REPORT_COLS); ax.set_ylim(0,1)
ax.set_title("Effet de la fuite de locuteur (DAIC-WOZ, niveau chunk)"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("rapport_figures/fig_fuite_locuteur.png", dpi=200); plt.show()

## Seuil de durée minimale

Les tours très courts sont nombreux dans le DAIC-WOZ. On balaie ici un seuil de durée minimale, de
0 à 10 secondes, pour voir s'il apporte quelque chose.

In [ ]:
thr = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0]
sweep = []
for t in thr:
    reps = 2 if (dur >= t).sum() > 15000 else 3   # moins de répétitions quand c'est lourd
    d = late_fusion_cv(Xc, yc, pid, dur, min_duration=t, n_repeats=reps)
    sweep.append({"seuil": t, "chunks": int((dur>=t).sum()),
                  "F1_macro": d.F1_macro.mean(), "F1_std": d.F1_macro.std(),
                  "UAR": d.UAR.mean(), "UAR_std": d.UAR.std(), "AUC": d.AUC.mean()})
    print(f"  seuil {t:>4}s : F1m={sweep[-1]['F1_macro']:.3f} UAR={sweep[-1]['UAR']:.3f}")
sweep_df = pd.DataFrame(sweep)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(13, 4.6))
a0.errorbar(sweep_df.seuil, sweep_df.F1_macro, yerr=sweep_df.F1_std, fmt="o-", color=C_A, label="F1 macro", capsize=3)
a0.errorbar(sweep_df.seuil, sweep_df.UAR, yerr=sweep_df.UAR_std, fmt="s-", color=C_HC, label="UAR", capsize=3)
a0.axhline(.5, ls=":", c="grey", label="hasard"); a0.set_xlabel("Seuil de durée min. (s)")
a0.set_ylabel("Score"); a0.set_title("DAIC-WOZ late fusion : performance vs seuil"); a0.legend(); a0.grid(alpha=.3)
a1.bar(sweep_df.seuil, sweep_df.chunks, width=.35, color=C_HC, alpha=.75)
a1.set_xlabel("Seuil de durée min. (s)"); a1.set_ylabel("Chunks conservés"); a1.grid(alpha=.3)
fig.tight_layout(); fig.savefig("rapport_figures/fig_seuil_duree.png", dpi=200); plt.show()

## Retrait des artefacts

L'ACP fait apparaître des groupes de segments sans parole, une ligne de silences et une zone de
souffles peu bruyants. On les retire ici en trois étapes, en réévaluant après chacune.

Le nettoyage manuel d'origine reposait sur des choix de clusters à l'œil. Cette version est
automatisée pour être reproductible, avec un retrait des segments de faible intensité, puis un
`LocalOutlierFactor` appliqué locuteur par locuteur, puis un DBSCAN.

In [ ]:
steps = []
# protocole : random_state=0, 5 plis x 10 répétitions
SEED_CASCADE = 0
def add_step(name, Xk, yk, pk, dk):
    d = late_fusion_cv(Xk, yk, pk, dk, min_duration=0.0, n_repeats=10, random_state=SEED_CASCADE)
    steps.append({"Étape": name, "chunks": len(Xk),
                  "F1_macro": f"{d.F1_macro.mean():.3f}", "UAR": f"{d.UAR.mean():.3f}"})
    print(f"  {name}: F1m={d.F1_macro.mean():.3f} UAR={d.UAR.mean():.3f} ({len(Xk)} chunks)")

# étape 0 : brut (tous les segments, aucun retrait d'artefact)
add_step("Brut (tous les segments)", Xc, yc, pid, dur)

# + retrait des silences/souffles : segments de faible loudness (index 10 = loudness_sma3_amean)
LOUD = 10
k0 = Xc[:, LOUD] >= np.percentile(Xc[:, LOUD], 20)   # on retire les 20 % les plus faibles
Xc, yc, pid, dur = Xc[k0], yc[k0], pid[k0], dur[k0]
add_step("+ retrait silences/souffles (loudness)", Xc, yc, pid, dur)

# + LOF PAR LOCUTEUR : pour chaque locuteur, on écarte ses 5 % de segments les plus atypiques
inl = np.ones(len(Xc), bool)
for p in np.unique(pid):
    idx = np.where(pid == p)[0]
    if len(idx) < 21: continue
    ok = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(
        StandardScaler().fit_transform(Xc[idx])) == 1
    inl[idx[~ok]] = False
Xc, yc, pid, dur = Xc[inl], yc[inl], pid[inl], dur[inl]
add_step("+ retrait points isolés (LOF par locuteur)", Xc, yc, pid, dur)

# + DBSCAN : on retire les points classés bruit (label -1) dans l'espace standardisé
keep = DBSCAN(eps=7, min_samples=10, n_jobs=-1).fit(StandardScaler().fit_transform(Xc)).labels_ != -1
Xc, yc, pid, dur = Xc[keep], yc[keep], pid[keep], dur[keep]
add_step("+ retrait bruit DBSCAN", Xc, yc, pid, dur)

# NB : seul le retrait des silences produit un gain (~1 centième d'UAR). Les deux étapes
# suivantes laissent la performance inchangée tout en retirant un quart des segments.
pd.DataFrame(steps)

## Le transfert d'un corpus à l'autre

Reste à tester si un modèle appris sur un corpus s'applique à l'autre. On entraîne sur l'un et on
teste sur l'autre, dans les deux sens.

In [ ]:
def cross_corpus_train_test(X_train, y_train, X_test, y_test, title):
    sc = StandardScaler().fit(Xtr)
    clf = SVC(class_weight="balanced", probability=True, random_state=RNG).fit(sc.transform(Xtr), ytr)
    p = clf.predict(sc.transform(Xte)); s = clf.predict_proba(sc.transform(Xte))[:, 1]
    m = evaluate(yte, p, s); print(title, {c: round(m[c],3) for c in REPORT_COLS})
    print("  confusion:\n", confusion_matrix(yte, p)); print()
    return m

m_d2a = cross_corpus_train_test(X_daic, y_daic, X_and, y_and, "DAIC-WOZ -> Androids")
m_a2d = cross_corpus_train_test(X_and, y_and, X_daic, y_daic, "Androids -> DAIC-WOZ")

# Récapitulatif

Quatre métriques sont rapportées. L'accuracy se lit avec prudence sur le DAIC-WOZ, dont les
classes sont très déséquilibrées, puisqu'y prédire systématiquement la classe majoritaire suffit
à la faire monter. L'UAR et la F1 macro n'ont pas ce défaut.

In [ ]:
recap = pd.DataFrame([
    intra_row("Androids Interview (early)", df_and),
    intra_row("DAIC-WOZ brut (early)",       df_daic),
    intra_row("DAIC-WOZ late fusion",        df_daic_late),
])
recap